# Importando as Bibliotecas

In [1]:
# instalei as principais bibliotecas para análise e formatação dos dados
%pip install pandas numpy matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


In [2]:
# importei a biblioteca pandas
import pandas as pd

# importei a biblioteca unicodedata para normalização de texto
import unicodedata

# criei a conexão do banco de dados na memória com o sql 
import sqlite3

conn = sqlite3.connect(":memory:")

# importei a biblioteca time para controle de tempo
import time

# importei a biblioteca requests para requisições HTTP
import requests

# importei a biblioteca numpy para operações numéricas
import numpy as np


# Tabela clientes

In [3]:
# carreguei os dados do dataset clientes e visualizei as primeiras linhas
df_clientes = pd.read_json('../dados_brutos/clientes_crm.json')

df_clientes.head()

,full_name,location,code,email
0,Femininos Oliveira Antunes,"Aratu (Candeias) , BA",1,femininos.oliveira.antunes@icloud.com
1,Fernanda Azevedo Soares Nunes Vieira,"PE , Recife",2,nunes.fernanda.soares.azevedo.vieira@outlook.com
2,Daniel Farias Ribeiro Teixeira,"Rio Grande,RS",3,farias.teixeira.daniel.ribeiro#gmail.com
3,Thiago Moreira,"AC , Rio Branco",4,thiago.moreira#gmail.com
4,Pedro Freitas,PA - Santarém Novo,5,pedro.freitas#icloud.com


In [4]:
# explorei a estrutura e as estatísticas do dataset de clientes
df_clientes.shape
df_clientes.info()
df_clientes.describe()

<class 'pandas.DataFrame'>
RangeIndex: 49 entries, 0 to 48
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   full_name  49 non-null     str  
 1   location   49 non-null     str  
 2   code       49 non-null     int64
 3   email      49 non-null     str  
dtypes: int64(1), str(3)
memory usage: 1.7 KB


,code
count,49.00000
mean,25.00000
std,14.28869
min,1.00000
25%,13.00000
50%,25.00000
75%,37.00000
max,49.00000


In [5]:
# verifiquei valores nulos no dataset de clientes
df_clientes.isnull().sum()

full_name    0
location     0
code         0
email        0
dtype: int64

In [6]:
# renomeei as colunas para padronzar os dados
df_clientes = df_clientes.rename(columns={
    'full_name': 'nome_completo',
    'location': 'localizacao',
    'code': 'id_cliente'
})

# verifiquei a nova condição
df_clientes.head()

,nome_completo,localizacao,id_cliente,email
0,Femininos Oliveira Antunes,"Aratu (Candeias) , BA",1,femininos.oliveira.antunes@icloud.com
1,Fernanda Azevedo Soares Nunes Vieira,"PE , Recife",2,nunes.fernanda.soares.azevedo.vieira@outlook.com
2,Daniel Farias Ribeiro Teixeira,"Rio Grande,RS",3,farias.teixeira.daniel.ribeiro#gmail.com
3,Thiago Moreira,"AC , Rio Branco",4,thiago.moreira#gmail.com
4,Pedro Freitas,PA - Santarém Novo,5,pedro.freitas#icloud.com


In [7]:
# organizei a ordem das colunas do dataset clientes
df_clientes = df_clientes[['id_cliente', 'nome_completo', 'localizacao', 'email']]

#Verificando a nova condição
df_clientes.head()

,id_cliente,nome_completo,localizacao,email
0,1,Femininos Oliveira Antunes,"Aratu (Candeias) , BA",femininos.oliveira.antunes@icloud.com
1,2,Fernanda Azevedo Soares Nunes Vieira,"PE , Recife",nunes.fernanda.soares.azevedo.vieira@outlook.com
2,3,Daniel Farias Ribeiro Teixeira,"Rio Grande,RS",farias.teixeira.daniel.ribeiro#gmail.com
3,4,Thiago Moreira,"AC , Rio Branco",thiago.moreira#gmail.com
4,5,Pedro Freitas,PA - Santarém Novo,pedro.freitas#icloud.com


In [8]:
# verifiquei a unicidade da chave id_cliente
df_clientes['id_cliente'].is_unique

True

In [9]:
# padronizei os nomes removendo os espaços e ajustando a capitalização
df_clientes['nome_completo'] = df_clientes['nome_completo'].str.strip()
df_clientes['nome_completo'] = df_clientes['nome_completo'].str.title()

In [10]:
# removi registros duplicados do dataset de clientes
df_clientes = df_clientes.drop_duplicates()

df_clientes.shape

(49, 4)

In [11]:
# verifiquei as colunas do dataset de clientes
df_clientes.columns

Index(['id_cliente', 'nome_completo', 'localizacao', 'email'], dtype='str')

In [12]:
# analisei os valores presentes na coluna localizacao para padronizar
df_clientes['localizacao'].unique()

<StringArray>
[         'Aratu (Candeias) , BA',                    'PE , Recife',
                  'Rio Grande,RS',                'AC , Rio Branco',
             'PA - Santarém Novo',      'Fortaleza do Tabocão , TO',
                    'PB/Cabedelo',                   'SE - Aracaju',
               'PB - João Pessoa',                  'Santarém / PA',
              'BA - Porto Seguro',      'TO , Fortaleza do Tabocão',
                  'PA / Santarém',               'AM , Itacoatiara',
        'Fortaleza do Tabocão,TO',                   'Fortaleza,CE',
                   'MS - Corumbá',                  'Santarém - PA',
                    'Maceió / AL',             'PA , Santarém Novo',
                  'AC,Rio Branco',                   'SE / Aracaju',
                    'Santos - SP',                    'Laguna / SC',
                'ES / São Mateus',                      'Manaus/AM',
                    'Salvador,BA',                  'PR , Antonina',
                   '

In [13]:
# defini um dicionário de estados
ufs = ['AC','AL','AP','AM','BA','CE','DF','ES','GO','MA','MT','MS','MG',
       'PA','PB','PR','PE','PI','RJ','RN','RS','RO','RR','SC','SP','SE','TO']

def limpar_loc(loc):
    if pd.isna(loc) or not loc.strip():
        return pd.Series([None, None])
    
    # normalizei e removi acentos
    loc = unicodedata.normalize('NFKD', str(loc).lower()).encode('ascii','ignore').decode('utf-8')
    
    # removi parênteses e padronizei separadores
    loc = loc.replace("(", "").replace(")", "")
    for sep in ['-', '/', '\\', ';']: loc = loc.replace(sep, ",")
    loc = loc.replace(",,",",").replace(", ",",").replace(" ,",",").strip()
    
    partes = [p.strip() for p in loc.split(",") if p.strip()]
    
    estado = None
    cidade = None
    
    # procurei UF em qualquer posição
    for i, p in enumerate(partes):
        if p.upper() in ufs:
            estado = p.upper()
            # defini que cidade é tudo que não é o estado
            cidade = " ".join([x.title() for j,x in enumerate(partes) if j != i])
            break
    
    # criei a condição de que caso não encontre UF, tentar assumir a última parte com 2 letras
    if estado is None and len(partes)>=2 and len(partes[-1])==2:
        estado = partes[-1].upper()
        cidade = " ".join(partes[:-1]).title()
    elif estado is None:
        cidade = " ".join(partes).title()
    
    return pd.Series([cidade, estado])

# apliquei no DataFrame
df_clientes[['cidade','estado']] = df_clientes['localizacao'].apply(limpar_loc)

print(df_clientes[['cidade','estado']].head(50))

                     cidade estado
0            Aratu Candeias     BA
1                    Recife     PE
2                Rio Grande     RS
3                Rio Branco     AC
4             Santarem Novo     PA
5      Fortaleza Do Tabocao     TO
6                  Cabedelo     PB
7                   Aracaju     SE
8               Joao Pessoa     PB
9                  Santarem     PA
10             Porto Seguro     BA
11     Fortaleza Do Tabocao     TO
12                 Santarem     PA
13              Itacoatiara     AM
14     Fortaleza Do Tabocao     TO
15                Fortaleza     CE
16                  Corumba     MS
17                 Santarem     PA
18                   Maceio     AL
19            Santarem Novo     PA
20               Rio Branco     AC
21                  Aracaju     SE
22                   Santos     SP
23                   Laguna     SC
24               Sao Mateus     ES
25                   Manaus     AM
26                 Salvador     BA
27                 A

In [14]:
# reorganizei as novas colunas do dataset de clientes
df_clientes = df_clientes [['id_cliente', 'nome_completo', 'email', 'cidade', 'estado']]

In [15]:
# normalizei a coluna email no dataset clientes
df_clientes['email'] = df_clientes['email'].str.replace('#', '@')

df_clientes[~df_clientes['email'].str.contains('@')]

#Verifiquei a nova condição
df_clientes.head()

,id_cliente,nome_completo,email,cidade,estado
0,1,Femininos Oliveira Antunes,femininos.oliveira.antunes@icloud.com,Aratu Candeias,BA
1,2,Fernanda Azevedo Soares Nunes Vieira,nunes.fernanda.soares.azevedo.vieira@outlook.com,Recife,PE
2,3,Daniel Farias Ribeiro Teixeira,farias.teixeira.daniel.ribeiro@gmail.com,Rio Grande,RS
3,4,Thiago Moreira,thiago.moreira@gmail.com,Rio Branco,AC
4,5,Pedro Freitas,pedro.freitas@icloud.com,Santarem Novo,PA


In [16]:
# chequei a tabela higienizada
df_clientes.info()
df_clientes.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 49 entries, 0 to 48
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id_cliente     49 non-null     int64
 1   nome_completo  49 non-null     str  
 2   email          49 non-null     str  
 3   cidade         49 non-null     str  
 4   estado         49 non-null     str  
dtypes: int64(1), str(4)
memory usage: 2.0 KB


id_cliente       0
nome_completo    0
email            0
cidade           0
estado           0
dtype: int64

In [17]:
# baixei os dados da tabela tratada
df_clientes.to_csv('../dados_tratados/clientes_crm_limpo.csv', index=False, encoding='utf-8-sig')

# Tabela de produtos

In [18]:
# carreguei os dados do dataset produtos e visualizei as primeiras linhas
df_produtos = pd.read_csv('../dados_brutos/produtos_raw.csv')

df_produtos.head()

,name,price,code,actual_category
0,Transponder AIS Maré Magnum,R$ 33122.52,1,ELETRONICOS
1,Transponder Furuno Marlin,R$ 13998.15,2,ELETRONICOS
2,Radar Furuno Pulse Leviathan,R$ 9024.19,3,E L E T R Ô N I C O S
3,Rádio AIS Hydro Tidal Zen,R$ 3381.88,4,Eletrunicos
4,Piloto Automático Furuno Storm,R$ 23669.01,5,Eletronicoz


In [19]:
# explorei a estrutura e as estatísticas do dataset de produtos
df_produtos.shape
df_produtos.info()
df_produtos.describe()

<class 'pandas.DataFrame'>
RangeIndex: 157 entries, 0 to 156
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   name             157 non-null    str  
 1   price            157 non-null    str  
 2   code             157 non-null    int64
 3   actual_category  157 non-null    str  
dtypes: int64(1), str(3)
memory usage: 5.0 KB


,code
count,157.000000
mean,76.210191
std,43.522766
min,1.000000
25%,39.000000
50%,75.000000
75%,114.000000
max,150.000000


In [20]:
# verifiquei valores nulos no dataset de produtos
df_produtos.isnull().sum()

name               0
price              0
code               0
actual_category    0
dtype: int64

In [21]:
# renomeei as colunas para padronização dos dados
df_produtos = df_produtos.rename(columns={
    'name': 'produto',
    'price': 'preco',
    'code': 'id_produto',
    'actual_category': 'categoria_real'

})

# verifiquei condição
df_produtos.head()

,produto,preco,id_produto,categoria_real
0,Transponder AIS Maré Magnum,R$ 33122.52,1,ELETRONICOS
1,Transponder Furuno Marlin,R$ 13998.15,2,ELETRONICOS
2,Radar Furuno Pulse Leviathan,R$ 9024.19,3,E L E T R Ô N I C O S
3,Rádio AIS Hydro Tidal Zen,R$ 3381.88,4,Eletrunicos
4,Piloto Automático Furuno Storm,R$ 23669.01,5,Eletronicoz


In [22]:
# organizei a ordem das colunas do dataset de produtos
df_produtos = df_produtos[['id_produto', 'produto', 'categoria_real', 'preco']]

# verifiquei condição
df_produtos.head()

,id_produto,produto,categoria_real,preco
0,1,Transponder AIS Maré Magnum,ELETRONICOS,R$ 33122.52
1,2,Transponder Furuno Marlin,ELETRONICOS,R$ 13998.15
2,3,Radar Furuno Pulse Leviathan,E L E T R Ô N I C O S,R$ 9024.19
3,4,Rádio AIS Hydro Tidal Zen,Eletrunicos,R$ 3381.88
4,5,Piloto Automático Furuno Storm,Eletronicoz,R$ 23669.01


In [23]:
# verifiquei unicidade da chave id_produto
df_produtos['id_produto'].is_unique

False

In [24]:
# utilizei da coluna "id_produto" para consultar a quantidade de produtos listados que estavam repetidos
df_produtos['id_produto'].duplicated().sum()

np.int64(7)

In [25]:
# para limpar a tabela, removi os prudutos duplicados com o método .drop_duplicates mantendo a primeira linha 
df_produtos = df_produtos.drop_duplicates(subset='id_produto', keep='first')

# verifiquei novamente a consistência da coluna "id_produto"
df_produtos['id_produto'].is_unique

True

In [26]:
# padronizei os nomes removendo os espaços e ajustando a capitalização
df_produtos['produto'] = df_produtos['produto'].str.strip()
df_produtos['produto'] = df_produtos['produto'].str.title()

Verificando a consistência da coluna "categoria_real":

In [27]:
# verifiquei todos os valores da coluna "categoria_real":
df_produtos['categoria_real'].unique()

<StringArray>
[          'ELETRONICOS', 'E L E T R Ô N I C O S',           'Eletrunicos',
           'Eletronicoz',           'eLeTrÔnIcOs',           'eletrônicos',
           'Eletrônicos',          'Eletroniscos',           'Eletronicos',
           'eletronicos',           'EletrônicoS',             'PROPULSAO',
             'Propulção',                  'Prop',            'Propulssão',
             'propulsao',     'P R O P U L S Ã O',             'pRoPuLsÃo',
             'Propulçao',              'Propução',             'propulsão',
             'Propulsam',             'PrOpUlSãO',             'Ancoragem',
             'AnCoRaGeM',             'Encoragem',            'Ancoraguem',
              'Ancorajm',             'AncorageM',     'A N C O R A G E M',
             'ANCORAGEM',             'aNcOrAgEm',             'Ancorajem',
             'ancoragem',             'Ancorajen']
Length: 35, dtype: str

In [28]:
# criei um dicionário para padronizar a coluna "categoria_real":
mapa_categorias = {
    # eletronicos
    'eletronicos': 'eletronicos',
    'eletrunicos': 'eletronicos',
    'eletroniscos': 'eletronicos',
    'eletronicoz': 'eletronicos',

    # propulsao
    'propulsao': 'propulsao',
    'propulcao': 'propulsao',
    'prop': 'propulsao',
    'propulssao': 'propulsao',
    'propucao': 'propulsao',
    'propulsam': 'propulsao',
    'propulsão': 'propulsao',

    # ancoragem
    'ancoragem': 'ancoragem',
    'encoragem': 'ancoragem',
    'ancoraguem': 'ancoragem',
    'ancorajm': 'ancoragem',
    'ancorajem': 'ancoragem',
    'ancorajen': 'ancoragem'
}

# reposicionei os valores da coluna "categoria _real" pelo mapa de categorias criado:
df_produtos['categoria_real'] = df_produtos['categoria_real'].replace(mapa_categorias)

In [29]:
# padronizei a formatação das strings:
df_produtos['categoria_real'] = (
    df_produtos['categoria_real']
    .str.lower()
    .str.strip()
    .str.replace(' ', '')
    .apply(lambda x: unicodedata.normalize('NFKD', x).encode('ascii', 'ignore').decode('utf-8'))
)

In [30]:
# verificando nova condição:
df_produtos['categoria_real'].unique()


<StringArray>
[ 'eletronicos',  'eletrunicos',  'eletronicoz', 'eletroniscos',
    'propulsao',    'propulcao',         'prop',   'propulssao',
     'propucao',    'propulsam',    'ancoragem',    'encoragem',
   'ancoraguem',     'ancorajm',    'ancorajem',    'ancorajen']
Length: 16, dtype: str

In [31]:
#Aqui eu formatei a coluna de "preco", uniformizando para o tipo numérico e reorganizando os caracteres
df_produtos['preco'] = (
    df_produtos['preco']
    .str.replace('R$', '', regex=False)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .str.strip()
    .astype(float)
)

In [32]:
# chequei a tabela higienizada
df_produtos.info()
df_produtos.isna().sum()

<class 'pandas.DataFrame'>
Index: 150 entries, 0 to 156
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_produto      150 non-null    int64  
 1   produto         150 non-null    str    
 2   categoria_real  150 non-null    str    
 3   preco           150 non-null    float64
dtypes: float64(1), int64(1), str(2)
memory usage: 5.9 KB


id_produto        0
produto           0
categoria_real    0
preco             0
dtype: int64

In [33]:
# baixei os dados da tabela tratada
df_produtos.to_csv('../dados_tratados/produtos_crm_limpo.csv', index=False, encoding='utf-8-sig')

# tabela de custos de importação

In [34]:
# carreguei os dados do dataset de custos e visualizei as primeiras linhas
df_custos = pd.read_json('../dados_brutos/custos_importacao.json')

df_custos.head()

,product_id,product_name,category,historic_data
0,1,Transponder AIS Maré Magnum,eletrônicos,"[{'start_date': '10/08/2016', 'usd_price': 105..."
1,2,Transponder Furuno Marlin,eletrônicos,"[{'start_date': '23/11/2017', 'usd_price': 432..."
2,3,Radar Furuno Pulse Leviathan,eletrônicos,"[{'start_date': '12/04/2016', 'usd_price': 254..."
3,4,Rádio AIS Hydro Tidal Zen,eletrônicos,"[{'start_date': '04/03/2016', 'usd_price': 909..."
4,5,Piloto Automático Furuno Storm,eletrônicos,"[{'start_date': '10/02/2016', 'usd_price': 600..."


In [35]:
# explorei a estrutura e as estatísticas do dataset de custos
df_custos.shape
df_custos.info()
df_custos.describe()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   product_id     150 non-null    int64 
 1   product_name   150 non-null    str   
 2   category       150 non-null    str   
 3   historic_data  150 non-null    object
dtypes: int64(1), object(1), str(2)
memory usage: 4.8+ KB


,product_id
count,150.000000
mean,75.500000
std,43.445368
min,1.000000
25%,38.250000
50%,75.500000
75%,112.750000
max,150.000000


In [36]:
# verifiquei valores nulos no dataset de produtos
df_custos.isnull().sum()

product_id       0
product_name     0
category         0
historic_data    0
dtype: int64

In [37]:
# renomeei as colunas para padronização dos dados
df_custos = df_custos.rename(columns={
    'product_id': 'id_produto',
    'product_name': 'produto',
    'category': 'categoria',
    'historic_data': 'historico_dados'

})

# verifiquei condição
df_custos.head()

,id_produto,produto,categoria,historico_dados
0,1,Transponder AIS Maré Magnum,eletrônicos,"[{'start_date': '10/08/2016', 'usd_price': 105..."
1,2,Transponder Furuno Marlin,eletrônicos,"[{'start_date': '23/11/2017', 'usd_price': 432..."
2,3,Radar Furuno Pulse Leviathan,eletrônicos,"[{'start_date': '12/04/2016', 'usd_price': 254..."
3,4,Rádio AIS Hydro Tidal Zen,eletrônicos,"[{'start_date': '04/03/2016', 'usd_price': 909..."
4,5,Piloto Automático Furuno Storm,eletrônicos,"[{'start_date': '10/02/2016', 'usd_price': 600..."


In [38]:
# verifiquei o tipo de dados na coluna historico_dados
df_custos['historico_dados'].apply(type).value_counts()

historico_dados
<class 'list'>    150
Name: count, dtype: int64

In [39]:
# expandi lista da coluna historico_dados em múltiplas linhas com o .explode
df_custos = df_custos.explode('historico_dados')

In [40]:
# extrai campos start_date e usd_price da coluna historico_dados
df_custos[['start_date', 'usd_price']] = df_custos['historico_dados'].apply(pd.Series)

In [41]:
# renomeei as colunas criadas
df_custos = df_custos.rename(columns={
    'start_date': 'data_inicial',
    'usd_price': 'preco_usd'
})

In [42]:
# eliminei a coluna "historico_dados"
df_custos = df_custos.drop(columns=['historico_dados'])

# verifiquei nova condição
df_custos.head()

,id_produto,produto,categoria,data_inicial,preco_usd
0,1,Transponder AIS Maré Magnum,eletrônicos,10/08/2016,10583.63
0,1,Transponder AIS Maré Magnum,eletrônicos,15/06/2018,8778.36
0,1,Transponder AIS Maré Magnum,eletrônicos,25/09/2018,8023.87
0,1,Transponder AIS Maré Magnum,eletrônicos,19/03/2019,8772.78
0,1,Transponder AIS Maré Magnum,eletrônicos,17/01/2020,7918.18


In [43]:
# verifiqeui a consistência da coluna id_produto
df_custos['id_produto'].is_unique

False

In [44]:
# removi as células duplicadas 
df_custos = df_custos.drop_duplicates(subset='id_produto', keep='first')

# verifiquei nova condição
df_custos['id_produto'].is_unique

True

In [45]:
# verifiquei a coluna categoria
df_custos['categoria'].unique()

<StringArray>
['eletrônicos', 'propulsão', 'ancoragem']
Length: 3, dtype: str

In [46]:
#verifiquei a coluna produto para possível tratamento 
df_custos['produto'].unique()

<StringArray>
[               'Transponder AIS Maré Magnum',
                  'Transponder Furuno Marlin',
               'Radar Furuno Pulse Leviathan',
                  'Rádio AIS Hydro Tidal Zen',
             'Piloto Automático Furuno Storm',
                     'Transponder AIS Vector',
                              'Radar AIS Zen',
                                'GPS AIS Zen',
                'Transponder AIS Titan Pulse',
 'Piloto Automático Simrad Titan Flux Magnum',
 ...
      'Boia de Arqueamento Bruce Nexus Abyss',
                 'Âncora Danforth Vector Evo',
         'Âncora Delta Hydra Kraken Velocity',
        'Boia de Arqueamento Bruce Barracuda',
            'Boia de Arqueamento Delta Nexus',
                     'Corrente Delta Vox Ion',
  'Corrente Danforth Force Leviathan Impulse',
          'Âncora Delta Force Barracuda Mako',
                   'Cabo de Nylon Bruce Core',
          'Cabo de Nylon Danforth Magnum Vox']
Length: 150, dtype: str

In [47]:
# removi os espaços da coluna "produto"
df_custos['produto'] = df_custos['produto'].str.strip()

In [48]:
# normalizei a coluna "categoria" removendo acentuação
def remover_acento(texto):
    return unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')

df_custos['categoria'] = df_custos['categoria'].apply(remover_acento)

In [49]:
# transformei o tipo de dado da coluna "data_inicial" para o tipo data
df_custos['data_inicial'] = pd.to_datetime(df_custos['data_inicial'], dayfirst=True)

In [50]:
# transformei o tipo de dado da coluna "preco_usd" para o tipo float
df_custos['preco_usd'] = df_custos['preco_usd'].astype(float)

In [51]:
# chequei a tabela higienizada
df_custos.info()
df_custos.isna().sum()

<class 'pandas.DataFrame'>
Index: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   id_produto    150 non-null    int64         
 1   produto       150 non-null    str           
 2   categoria     150 non-null    str           
 3   data_inicial  150 non-null    datetime64[us]
 4   preco_usd     150 non-null    float64       
dtypes: datetime64[us](1), float64(1), int64(1), str(2)
memory usage: 7.0 KB


id_produto      0
produto         0
categoria       0
data_inicial    0
preco_usd       0
dtype: int64

In [52]:
# baixei os dados da tabela tratada
df_custos.to_csv('../dados_tratados/custos_crm_limpo.csv', index=False, encoding='utf-8-sig')

# tabela de vendas 2023/2024

In [53]:
# carreguei os dados do dataset de vendas e visualizei as primeiras linhas
df_vendas = pd.read_csv('../dados_brutos/vendas_2023_2024.csv')

df_vendas.head()

,id,id_client,id_product,qtd,total,sale_date
0,0,42,105,11,3405.0,2023-09-10
1,1,3,136,9,16873.9,15-09-2024
2,2,25,139,7,9475.3,2024-08-13
3,4,20,23,5,55893.0,2023-02-03
4,5,8,57,4,451403.9,2024-02-12


In [54]:
# explorei a estrutura e as estatísticas do dataset de vendas
df_vendas.shape
df_vendas.info()
df_vendas.describe()

<class 'pandas.DataFrame'>
RangeIndex: 9895 entries, 0 to 9894
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          9895 non-null   int64  
 1   id_client   9895 non-null   int64  
 2   id_product  9895 non-null   int64  
 3   qtd         9895 non-null   int64  
 4   total       9895 non-null   float64
 5   sale_date   9895 non-null   str    
dtypes: float64(1), int64(4), str(1)
memory usage: 464.0 KB


,id,id_client,id_product,qtd,total
count,9895.000000,9895.000000,9895.000000,9895.000000,9.895000e+03
mean,5000.755533,24.874583,75.255786,8.015260,2.637978e+05
std,2887.000000,14.177715,43.533397,4.301723,3.900072e+05
min,0.000000,1.000000,1.000000,1.000000,2.945000e+02
25%,2501.500000,13.000000,37.000000,4.000000,2.313820e+04
50%,4999.000000,25.000000,74.000000,8.000000,8.222500e+04
75%,7505.500000,37.000000,114.000000,12.000000,3.390945e+05
max,9999.000000,49.000000,150.000000,15.000000,2.222973e+06


In [55]:
# verifiquei valores nulos no dataset de vendas
df_vendas.isnull().sum()

id            0
id_client     0
id_product    0
qtd           0
total         0
sale_date     0
dtype: int64

In [56]:
# verifiquei possíveis outliers na coluna total
Q1 = df_vendas['total'].quantile(0.25)
Q3 = df_vendas['total'].quantile(0.75)
IQR = Q3 - Q1


limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers = df_vendas[(df_vendas['total'] < limite_inferior) | (df_vendas['total'] > limite_superior)]
print(f"Quantidade de outliers: {len(outliers)}")
outliers.head()

Quantidade de outliers: 1018


,id,id_client,id_product,qtd,total,sale_date
15,17,43,91,11,1409558.70,2023-09-05
26,28,41,60,12,1239762.35,2023-03-20
29,31,41,91,13,1665842.10,2023-03-20
37,39,13,71,8,989322.40,2024-10-16
42,44,39,81,8,928328.60,2024-12-03


In [57]:
# renomeei as colunas para padronização dos dados
df_vendas = df_vendas.rename(columns={
    'id_client': 'id_cliente',
    'id_product': 'id_produto',
    'sale_date': 'data_venda'
})

# verifiquei nova condição
df_vendas.head()

,id,id_cliente,id_produto,qtd,total,data_venda
0,0,42,105,11,3405.0,2023-09-10
1,1,3,136,9,16873.9,15-09-2024
2,2,25,139,7,9475.3,2024-08-13
3,4,20,23,5,55893.0,2023-02-03
4,5,8,57,4,451403.9,2024-02-12


In [58]:
# organizei a ordem das colunas do dataset de vendas
df_vendas = df_vendas[['id', 'id_cliente', 'id_produto', 'data_venda', 'qtd', 'total']]

#Verificando condição
df_vendas.head()

,id,id_cliente,id_produto,data_venda,qtd,total
0,0,42,105,2023-09-10,11,3405.0
1,1,3,136,15-09-2024,9,16873.9
2,2,25,139,2024-08-13,7,9475.3
3,4,20,23,2023-02-03,5,55893.0
4,5,8,57,2024-02-12,4,451403.9


In [59]:
# verifiquei a consistência das colunas de identificação
df_vendas.duplicated(subset=['id', 'id_produto', 'id_cliente'])

0       False
1       False
2       False
3       False
4       False
        ...  
9890    False
9891    False
9892    False
9893    False
9894    False
Length: 9895, dtype: bool

In [60]:
# formatei o tipo da coluna "data_venda" para data
df_vendas['data_venda'] = pd.to_datetime(
    df_vendas['data_venda'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)

In [61]:
# baixei os dados da tabela tratada
df_vendas.info()
df_vendas.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 9895 entries, 0 to 9894
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   id          9895 non-null   int64         
 1   id_cliente  9895 non-null   int64         
 2   id_produto  9895 non-null   int64         
 3   data_venda  9895 non-null   datetime64[us]
 4   qtd         9895 non-null   int64         
 5   total       9895 non-null   float64       
dtypes: datetime64[us](1), float64(1), int64(4)
memory usage: 464.0 KB


id            0
id_cliente    0
id_produto    0
data_venda    0
qtd           0
total         0
dtype: int64

In [62]:
# baixei os dados da tabela tratada
df_vendas.to_csv('../dados_tratados/vendas_crm_limpo.csv', index=False, encoding='utf-8-sig')

# SQL

In [63]:
# converti as tabelas para o sqlite
df_clientes = pd.read_csv("../dados_tratados/clientes_crm_limpo.csv")
df_vendas = pd.read_csv("../dados_tratados/vendas_crm_limpo.csv")
df_produtos = pd.read_csv("../dados_tratados/produtos_crm_limpo.csv")
df_custos = pd.read_csv("../dados_tratados/custos_crm_limpo.csv")

In [64]:
df_clientes.to_sql('clientes_crm_limpo', conn, if_exists='replace', index=False)
df_vendas.to_sql('vendas_crm_limpo', conn, if_exists='replace', index=False)
df_produtos.to_sql('produtos_crm_limpo', conn, if_exists='replace', index=False)
df_custos.to_sql('custos_crm_limpo', conn, if_exists='replace', index=False)

150

### consultas da tabela clientes:

In [65]:
# verifiquei a integridade dos dados convertidos na tabela
query = """
SELECT * FROM clientes_crm_limpo
"""

pd.read_sql(query, conn)

,id_cliente,nome_completo,email,cidade,estado
0,1,Femininos Oliveira Antunes,femininos.oliveira.antunes@icloud.com,Aratu Candeias,BA
1,2,Fernanda Azevedo Soares Nunes Vieira,nunes.fernanda.soares.azevedo.vieira@outlook.com,Recife,PE
2,3,Daniel Farias Ribeiro Teixeira,farias.teixeira.daniel.ribeiro@gmail.com,Rio Grande,RS
3,4,Thiago Moreira,thiago.moreira@gmail.com,Rio Branco,AC
4,5,Pedro Freitas,pedro.freitas@icloud.com,Santarem Novo,PA
5,6,Antônia Coelho Pinheiro Peixoto Cavalcanti,coelho.pinheiro.peixoto.antônia.cavalcanti@aol...,Fortaleza Do Tabocao,TO
6,7,Bianca Barros Rocha Torres Siqueira,torres.barros.rocha.bianca.siqueira@aol.com,Cabedelo,PB
7,8,Luiz Alves Pimentel,pimentel.alves.luiz@outlook.com,Aracaju,SE
8,9,Lucas Guedes Cunha Lopes,lucas.lopes.guedes.cunha@tutanota.com,Joao Pessoa,PB
9,10,Débora Paiva,paiva.débora@gmx.com,Santarem,PA


In [66]:
# chequei a quantidade de clientes por estado
query = """
SELECT 
    estado,
    COUNT(*) AS total_clientes
FROM clientes_crm_limpo
GROUP BY estado
ORDER BY total_clientes DESC;
"""
pd.read_sql(query,conn)

,estado,total_clientes
0,PA,8
1,BA,5
2,TO,4
3,SE,3
4,PE,3
5,PB,3
6,MA,3
7,CE,3
8,AM,3
9,RS,2


In [67]:
# verifiquei a quantidade de clientes por cidade
query = """
SELECT
    cidade,
    COUNT(*) AS total_clientes
FROM 
    clientes_crm_limpo
GROUP BY 
    cidade
ORDER BY 
    total_clientes DESC;
"""

pd.read_sql(query, conn)

,cidade,total_clientes
0,Santarem,4
1,Fortaleza Do Tabocao,4
2,Fortaleza,3
3,Aracaju,3
4,Suape Ipojuca,2
5,Santarem Novo,2
6,Rio Grande,2
7,Rio Branco,2
8,Porto Seguro,2
9,Joao Pessoa,2


### Consultas da tabela produtos:

In [68]:
# verifiquei a integridade dos dados convertidos na tabela
query = """
SELECT * FROM produtos_crm_limpo
""" 
pd.read_sql(query, conn)

,id_produto,produto,categoria_real,preco
0,1,Transponder Ais Maré Magnum,eletronicos,3312252.0
1,2,Transponder Furuno Marlin,eletronicos,1399815.0
2,3,Radar Furuno Pulse Leviathan,eletronicos,902419.0
3,4,Rádio Ais Hydro Tidal Zen,eletrunicos,338188.0
4,5,Piloto Automático Furuno Storm,eletronicoz,2366901.0
...,...,...,...,...
145,146,Corrente Delta Vox Ion,ancoragem,49598.0
146,147,Corrente Danforth Force Leviathan Impulse,ancoraguem,303008.0
147,148,Âncora Delta Force Barracuda Mako,ancorajem,478556.0
148,149,Cabo De Nylon Bruce Core,ancoragem,116362.0


In [69]:
# chequei o preço médio por categoria
query = """
SELECT 
    categoria_real,
    ROUND(AVG(preco), 2) AS preco_medio
FROM produtos_crm_limpo
GROUP BY categoria_real
ORDER BY preco_medio DESC;
"""

df_produtos = pd.read_sql(query, conn)

In [70]:
df_produtos['preco_formatado'] = df_produtos['preco_medio'].apply(
    lambda x: f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)

df_produtos

,categoria_real,preco_medio,preco_formatado
0,prop,9433291.25,"R$ 9.433.291,25"
1,propulssao,9307477.00,"R$ 9.307.477,00"
2,propulsao,7939652.25,"R$ 7.939.652,25"
3,propucao,7462211.86,"R$ 7.462.211,86"
4,propulsam,6314575.50,"R$ 6.314.575,50"
5,propulcao,5085778.63,"R$ 5.085.778,63"
6,eletroniscos,2349485.50,"R$ 2.349.485,50"
7,eletronicoz,1709546.57,"R$ 1.709.546,57"
8,eletronicos,1696603.47,"R$ 1.696.603,47"
9,eletrunicos,1331301.40,"R$ 1.331.301,40"


In [71]:
# verifiquei o vlor total das categorias para analisar como essa média ficou distribuida
query = """
SELECT
    categoria_real,
    SUM(preco) AS valor_total
FROM produtos_crm_limpo
GROUP BY categoria_real
ORDER BY valor_total DESC;
"""
df_produtos = pd.read_sql(query, conn)

In [72]:
df_produtos['preco_formatado'] = df_produtos['valor_total'].apply(
    lambda x: f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)

df_produtos

,categoria_real,valor_total,preco_formatado
0,propulsao,190551654.0,"R$ 190.551.654,00"
1,eletronicos,61077725.0,"R$ 61.077.725,00"
2,propucao,52235483.0,"R$ 52.235.483,00"
3,propulssao,46537385.0,"R$ 46.537.385,00"
4,propulcao,40686229.0,"R$ 40.686.229,00"
5,prop,37733165.0,"R$ 37.733.165,00"
6,propulsam,12629151.0,"R$ 12.629.151,00"
7,eletronicoz,11966826.0,"R$ 11.966.826,00"
8,ancoragem,6799025.0,"R$ 6.799.025,00"
9,eletrunicos,6656507.0,"R$ 6.656.507,00"


In [73]:
# ordenei os produtos mais caros
query = """
SELECT 
    produto,
    preco
FROM produtos_crm_limpo
ORDER BY preco DESC
LIMIT 10;
"""

df_produtos = pd.read_sql(query, conn)

In [74]:
df_produtos['preco_formatado'] = df_produtos['preco'].apply(
    lambda x: f"R$ {x:,.2f}".replace(",","x").replace('.', ",").replace("x",".")
)
df_produtos

,produto,preco,preco_formatado
0,Motor Diesel Honda Aero 205Hp,14819823.0,"R$ 14.819.823,00"
1,Motor De Popa Torqeedo Core Hydra Flux 162Hp,14315993.0,"R$ 14.315.993,00"
2,Motor Elétrico Torqeedo Ion Orca Vox 186Hp,13995767.0,"R$ 13.995.767,00"
3,Motor De Popa Tohatsu Evo 168Hp,13533507.0,"R$ 13.533.507,00"
4,Motor Elétrico Tohatsu Zenith Oceanic 113Hp,13488602.0,"R$ 13.488.602,00"
5,Motor Diesel Yanmar Dash Nitro 184Hp,13480388.0,"R$ 13.480.388,00"
6,Motor Elétrico Torqeedo Pulse 300Hp,13017406.0,"R$ 13.017.406,00"
7,Motor De Popa Volvo Maré 69Hp,12922384.0,"R$ 12.922.384,00"
8,Motor Elétrico Torqeedo Barracuda Magnum Helix...,12214849.0,"R$ 12.214.849,00"
9,Motor Elétrico Honda Mako Axis 131Hp,12176156.0,"R$ 12.176.156,00"


In [75]:
# ordenei os produtos mais baratos
query = """
SELECT 
    produto,
    preco
FROM produtos_crm_limpo
ORDER BY preco ASC
LIMIT 10;
"""

df_produtos = pd.read_sql(query, conn)

In [76]:
df_produtos['preco_formatado'] = df_produtos['preco'].apply(
    lambda x: f"R$ {x:,.2f}".replace(",","x").replace('.', ",").replace("x",".")
)
df_produtos

,produto,preco,preco_formatado
0,Cabo De Nylon Delta Vortex,4634.0,"R$ 4.634,00"
1,Cabo De Nylon Danforth Evo Aqua Mako,6227.0,"R$ 6.227,00"
2,Cabo De Nylon Bruce Flux Hydro,19735.0,"R$ 19.735,00"
3,Cabo De Nylon Danforth Prime,30954.0,"R$ 30.954,00"
4,Boia De Arqueamento Bruce Nexus Abyss,47818.0,"R$ 47.818,00"
5,Corrente Delta Vox Ion,49598.0,"R$ 49.598,00"
6,Radar Simrad Boost,77471.0,"R$ 77.471,00"
7,Boia De Arqueamento Danforth Tidal Zen Swift,79929.0,"R$ 79.929,00"
8,Âncora Danforth Vector Evo,82641.0,"R$ 82.641,00"
9,Cabo De Nylon Delta Force Magnum Leviathan,83059.0,"R$ 83.059,00"


In [77]:
# verifiquei o produto mais caro de cata categoria
query = """
SELECT
    categoria_real,
    produto, 
    MAX(preco) AS maior_preco
FROM produtos_crm_limpo
GROUP BY categoria_real;
""" 
df_produtos = pd.read_sql(query, conn)


In [78]:
df_produtos['preco_formatado'] = df_produtos['maior_preco'].apply(
    lambda x: f"R$ {x:,.2f}".replace(",","x").replace('.', ",").replace("x",".")
)
df_produtos

,categoria_real,produto,maior_preco,preco_formatado
0,ancoragem,Âncora Delta Storm Vox,471761.0,"R$ 471.761,00"
1,ancoraguem,Boia De Arqueamento Delta Peak Boost Thrust,418305.0,"R$ 418.305,00"
2,ancorajem,Âncora Delta Force Barracuda Mako,478556.0,"R$ 478.556,00"
3,ancorajen,Cabo De Nylon Danforth Titan Poseidon,164451.0,"R$ 164.451,00"
4,ancorajm,Âncora Danforth Nitro Titan Vox,438487.0,"R$ 438.487,00"
5,eletronicos,Gps Furuno Swift Leviathan Poseidon,3623191.0,"R$ 3.623.191,00"
6,eletronicoz,Transponder Furuno Force Ion,3945286.0,"R$ 3.945.286,00"
7,eletroniscos,Rádio Furuno Zen Swift,3581108.0,"R$ 3.581.108,00"
8,eletrunicos,Rádio Simrad Orca,3651034.0,"R$ 3.651.034,00"
9,encoragem,Boia De Arqueamento Bruce Marlin Hydra,465072.0,"R$ 465.072,00"


### Consulta da tabela de custos:

In [79]:
# verifiquei a integridade dos dados convertidos na tabela
query = """
SELECT * FROM custos_crm_limpo
"""

pd.read_sql(query, conn)

,id_produto,produto,categoria,data_inicial,preco_usd
0,1,Transponder AIS Maré Magnum,eletronicos,2016-08-10,10583.63
1,2,Transponder Furuno Marlin,eletronicos,2017-11-23,4325.09
2,3,Radar Furuno Pulse Leviathan,eletronicos,2016-04-12,2549.21
3,4,Rádio AIS Hydro Tidal Zen,eletronicos,2016-03-04,909.55
4,5,Piloto Automático Furuno Storm,eletronicos,2016-02-10,6006.45
...,...,...,...,...,...
145,146,Corrente Delta Vox Ion,ancoragem,2016-05-03,139.54
146,147,Corrente Danforth Force Leviathan Impulse,ancoragem,2016-05-13,864.87
147,148,Âncora Delta Force Barracuda Mako,ancoragem,2017-06-06,1458.52
148,149,Cabo de Nylon Bruce Core,ancoragem,2016-02-19,287.41


In [80]:
# verifiquei o preço médio por categoria
query = """
SELECT 
    categoria,
    ROUND(AVG(preco_usd)) AS preco_medio
FROM custos_crm_limpo
GROUP BY categoria
ORDER BY preco_medio DESC
"""
df_custos = pd.read_sql(query, conn)

In [81]:
df_custos['preco_formatado'] = df_custos['preco_medio'].apply(
    lambda x: f"$ {x:,.2f}"
)

df_custos 

,categoria,preco_medio,preco_formatado
0,propulsao,23932.0,"$ 23,932.00"
1,eletronicos,5108.0,"$ 5,108.00"
2,ancoragem,757.0,$ 757.00


In [82]:
# verifiquei o vlor total das categorias para analisar como essa média ficou distribuida
query = """
SELECT
    categoria,
    SUM(preco_usd) AS custo_total
FROM custos_crm_limpo
GROUP BY categoria
ORDER BY custo_total DESC;
"""
df_custos = pd.read_sql(query, conn)

In [83]:
df_custos['custo_formatado'] = df_custos['custo_total'].apply(
    lambda x: f"$ {x:,.2f}"
)

df_custos 

,categoria,custo_total,custo_formatado
0,propulsao,1196580.84,"$ 1,196,580.84"
1,eletronicos,255404.82,"$ 255,404.82"
2,ancoragem,37871.47,"$ 37,871.47"


In [84]:
# verifiqeui os produtos com custo mais alto
query = """
SELECT produto, MAX(preco_usd) as preco_usd
FROM custos_crm_limpo
GROUP BY produto
ORDER BY preco_usd DESC
LIMIT 10
"""
df_custos = pd.read_sql(query, conn)

In [85]:
df_custos['custo_formatado'] = df_custos['preco_usd'].apply(
    lambda x: f"$ {x:,.2f}"
)

df_custos 

,produto,preco_usd,custo_formatado
0,Motor Elétrico Torqeedo Ion Orca Vox 186HP,45881.74,"$ 45,881.74"
1,Motor de Popa Tohatsu Evo 168HP,41626.19,"$ 41,626.19"
2,Motor de Popa Torqeedo Core Hydra Flux 162HP,41323.15,"$ 41,323.15"
3,Motor Elétrico Torqeedo Pulse 300HP,40725.21,"$ 40,725.21"
4,Motor Elétrico Tohatsu Zenith Oceanic 113HP,38934.89,"$ 38,934.89"
5,Motor Diesel Honda Aero 205HP,36971.92,"$ 36,971.92"
6,Motor Elétrico Honda Mako Axis 131HP,36569.43,"$ 36,569.43"
7,Motor Diesel Yanmar Dash Nitro 184HP,36265.87,"$ 36,265.87"
8,Motor de Popa Honda Vector Kinetic 174HP,36236.52,"$ 36,236.52"
9,Motor Elétrico Torqeedo Barracuda Magnum Helix...,34870.68,"$ 34,870.68"


In [86]:
# verifiquei os produtos com custo mais baixo
query = """
SELECT 
    produto,
    preco_usd
FROM custos_crm_limpo
ORDER BY preco_usd ASC
LIMIT 10;
"""

df_custos = pd.read_sql(query, conn)

In [87]:
df_custos['custo_formatado'] = df_custos['preco_usd'].apply(
    lambda x: f"$ {x:,.2f}"
)

df_custos 

,produto,preco_usd,custo_formatado
0,Cabo de Nylon Danforth Prime,74.49,$ 74.49
1,Cabo de Nylon Delta Vortex,132.79,$ 132.79
2,Corrente Delta Vox Ion,139.54,$ 139.54
3,Cabo de Nylon Danforth Evo Aqua Mako,196.71,$ 196.71
4,Boia de Arqueamento Danforth Tidal Zen Swift,238.81,$ 238.81
5,Âncora Danforth Vector Evo,250.09,$ 250.09
6,Boia de Arqueamento Danforth Core Vortex,257.63,$ 257.63
7,Cabo de Nylon Delta Force Magnum Leviathan,258.29,$ 258.29
8,Cabo de Nylon Bruce Nexus,263.57,$ 263.57
9,Cabo de Nylon Bruce Core,287.41,$ 287.41


### Consultas da tabela de vendas 2023/2024

In [88]:
# verifiquei a integridade dos dados convertidos na tabela
query  = """
SELECT * FROM vendas_crm_limpo
"""
pd.read_sql(query, conn)

,id,id_cliente,id_produto,data_venda,qtd,total
0,0,42,105,2023-10-09,11,3405.00
1,1,3,136,2024-09-15,9,16873.90
2,2,25,139,2024-08-13,7,9475.30
3,4,20,23,2023-03-02,5,55893.00
4,5,8,57,2024-12-02,4,451403.90
...,...,...,...,...,...,...
9890,9995,30,139,2023-03-11,6,8549.00
9891,9996,9,111,2023-09-17,7,28497.15
9892,9997,38,123,2023-06-20,2,5276.30
9893,9998,33,97,2024-10-23,6,771409.50


In [89]:
df_produtos.rename(columns={'id':'id_produto'}, inplace=True)

In [90]:
# uni o dataset de vendas com a tabela de produtos
df_produtos_original = pd.read_csv(
    r"C:\Users\Rangel\OneDrive\Documentos\lh_nautical\dados_tratados\produtos_crm_limpo.csv"
)

In [91]:
df_vendas_produtos = df_vendas.merge(
    df_produtos_original,
    on='id_produto',
    how='left'
)

df_vendas_produtos.head()

,id,id_cliente,id_produto,data_venda,qtd,total,produto,categoria_real,preco
0,0,42,105,2023-10-09,11,3405.0,Cabo De Nylon Danforth Prime,ancoraguem,30954.0
1,1,3,136,2024-09-15,9,16873.9,Cabo De Nylon Bruce Flux Hydro,ancorajen,19735.0
2,2,25,139,2024-08-13,7,9475.3,Boia De Arqueamento Danforth Torque,ancorajm,142488.0
3,4,20,23,2023-03-02,5,55893.0,Piloto Automático Furuno Torque Peak,eletroniscos,1117863.0
4,5,8,57,2024-12-02,4,451403.9,Motor De Popa Honda Vector Kinetic 174Hp,propulsao,11879057.0


In [92]:
df_vendas_produtos.to_sql('vendas_produtos', conn, index=False, if_exists='replace')

9895

In [93]:
# verifiquei os produtos que mais geraramm receita para a empresa
query = """
SELECT 
    produto,
    SUM(total) AS total_vendas
FROM vendas_produtos
GROUP BY produto
ORDER BY total_vendas DESC
LIMIT 10;
"""

melhores_produtos = pd.read_sql(query, conn)

In [94]:
melhores_produtos['preco_formatado'] = melhores_produtos['total_vendas'].apply(
    lambda x: f"R$ {x:,.2f}".replace(",", "x").replace(".", ",").replace("x", ".")
)

melhores_produtos.head(10)

,produto,total_vendas,preco_formatado
0,Motor Diesel Honda Aero 205Hp,83539339.40,"R$ 83.539.339,40"
1,Motor Elétrico Torqeedo Pulse 300Hp,81567066.65,"R$ 81.567.066,65"
2,Motor De Popa Torqeedo Core Hydra Flux 162Hp,69554254.80,"R$ 69.554.254,80"
3,Motor Elétrico Torqeedo Ion Orca Vox 186Hp,68817185.90,"R$ 68.817.185,90"
4,Motor De Popa Volvo Maré 69Hp,67332086.05,"R$ 67.332.086,05"
5,Motor Elétrico Tohatsu Zenith Oceanic 113Hp,66829268.70,"R$ 66.829.268,70"
6,Motor De Popa Yamaha Evo Dash 155Hp,65859716.10,"R$ 65.859.716,10"
7,Motor Elétrico Torqeedo Barracuda Magnum Helix...,64195127.60,"R$ 64.195.127,60"
8,Motor De Popa Volvo Hydro Dash 256Hp,63057815.65,"R$ 63.057.815,65"
9,Motor Diesel Volvo Flow Oceanic 259Hp,61224375.00,"R$ 61.224.375,00"


In [95]:
# verifiquei a ordem das categorias da maior para a menor receita
query = """
SELECT 
    categoria_real,
    SUM(total) AS total_vendas
FROM vendas_produtos
GROUP BY categoria_real
ORDER BY total_vendas DESC;
"""
top_categoria = pd.read_sql(query, conn)

In [96]:
top_categoria['preco_formatado'] = top_categoria['total_vendas'].apply(
    lambda x: f"R$ {x:,.2f}".replace(",", "x").replace(".", ",").replace("x", ".")
)
top_categoria.head(10)


,categoria_real,total_vendas,preco_formatado
0,propulsao,9.998226e+08,"R$ 999.822.581,20"
1,propucao,3.236797e+08,"R$ 323.679.679,15"
2,eletronicos,3.188405e+08,"R$ 318.840.494,00"
3,propulcao,2.683400e+08,"R$ 268.339.968,60"
4,propulssao,2.403138e+08,"R$ 240.313.760,20"
5,prop,1.862893e+08,"R$ 186.289.302,15"
6,eletronicoz,8.793816e+07,"R$ 87.938.155,90"
7,propulsam,5.782885e+07,"R$ 57.828.849,55"
8,ancoragem,3.882231e+07,"R$ 38.822.311,65"
9,eletrunicos,3.446010e+07,"R$ 34.460.101,35"


In [97]:
# verifiquei o faturamento da empresa por ano
query ="""
SELECT 
    strftime('%Y', data_venda) AS ano,
    SUM(total) AS total_vendas
FROM vendas_produtos
GROUP BY ano
ORDER BY ano;
"""
vendas_ano = pd.read_sql(query, conn)


In [98]:
vendas_ano['preco_formatado'] = vendas_ano['total_vendas'].apply(
    lambda x: f"R$ {x:,.2f}".replace(",", "x").replace(".", ",").replace("x", ".")
)
vendas_ano.head(10)


,ano,total_vendas,preco_formatado
0,2023,1.288827e+09,"R$ 1.288.827.294,55"
1,2024,1.321452e+09,"R$ 1.321.452.216,15"


# Questões do desafio:

Questões 1.1 e 1.2:

In [99]:
df_vendas.to_sql('vendas_2023_2024', conn, index=False, if_exists='replace')

9895

In [100]:
# Recarregamento da base original
df_vendas = pd.read_csv('../dados_brutos/vendas_2023_2024.csv')

df_vendas.to_sql('vendas_2023_2024', conn, index=False, if_exists='replace')

9895

In [101]:
query = """
SELECT
    COUNT(*) AS num_linhas,
    6 AS num_colunas,

    MIN(DATE(sale_date)) AS data_min,
    MAX(DATE(sale_date)) AS data_max,

    MIN(total) AS valor_min,
    MAX(total) AS valor_max,
    AVG(total) AS valor_medio

FROM vendas_2023_2024
"""
pd.read_sql(query, conn)

,num_linhas,num_colunas,data_min,data_max,valor_min,valor_max,valor_medio
0,9895,6,2023-01-01,2024-12-31,294.5,2222973.0,263797.828267


Questão 1.3:

A coluna total apresenta aproximadamente 1.018 valores divergentes em relação à média, representando cerca de 10% do total de 9.895 registros. Esses valores podem indicar a presença de outliers, possivelmente associados a descontos, promoções ou vendas de maior volume, sendo necessário avaliar seu impacto conforme o objetivo da análise
 A tabela Vendas_2023_2024.csv não apresentou valores nulos durante a etapa de verificação, indicando consistência no preenchimento dos dados.
 No entanto, a estrutura da base não se encontra adequada para consultas estruturadas e análises mais robustas. A coluna sale_date, por exemplo, está armazenada como string e apresenta formatos de data inconsistentes, o que exige a padronização para o tipo datetime.
 Dessa forma, torna-se necessária a realização de um processo de higienização e transformação dos dados, visando garantir maior qualidade, consistência e confiabilidade para as análises subsequentes.

Questão 2.1:
Markdown produtos/ Verificando a consistência da coluna "categoria_real"

Questão 2.2: 
7 produtos removidos

Questão 3.1 e 3.2:

In [102]:
df_custos = pd.read_json('../dados_brutos/custos_importacao.json')

# Desmembrando "historic_data"
df_custos = df_custos.explode('historic_data')

# Renomeando colunas
df_custos = df_custos.rename(columns={
    'product_id': 'id_produto',
    'product_name': 'produto',
    'category': 'categoria',
    'historic_data': 'historico_dados'
})

# Extraindo start_date e usd_price do dicionário
df_custos[['data_inicial', 'preco_usd']] = df_custos['historico_dados'].apply(pd.Series)
df_custos = df_custos.drop(columns=['historico_dados'])

# Normalizando categoria
def remover_acento(texto):
    return unicodedata.normalize('NFKD', str(texto)).encode('ascii', 'ignore').decode('utf-8')

df_custos['categoria'] = df_custos['categoria'].apply(remover_acento)
df_custos['data_inicial'] = pd.to_datetime(df_custos['data_inicial'], dayfirst=True)
df_custos['preco_usd'] = df_custos['preco_usd'].astype(float)

print(f"Total de entradas: {len(df_custos)}") 

df_custos.to_csv('../dados_tratados/custos_crm_limpo.csv', index=False, encoding='utf-8-sig')

Total de entradas: 1260


Questão 4.1:

In [103]:
# lendo os arquivos de custos e vendas
vendas  = pd.read_csv('../dados_tratados/vendas_crm_limpo.csv')
custos  = pd.read_csv('../dados_tratados/custos_crm_limpo.csv')

vendas['data_venda'] = pd.to_datetime(vendas['data_venda'])
custos['data_inicial'] = pd.to_datetime(custos['data_inicial'])

# buscando câmbio USD→BRL com API do Banco Central para cada data de venda
def buscar_cambio(data: str) -> float:
    """Retorna a taxa de venda (média) do dólar para uma data no formato YYYY-MM-DD."""
    d = pd.to_datetime(data).strftime('%m-%d-%Y')
    url = (
        f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
        f"CotacaoDolarDia(dataCotacao=@dataCotacao)"
        f"?@dataCotacao='{d}'&$top=1&$format=json&$select=cotacaoVenda"
    )
    try:
        r = requests.get(url, timeout=10)
        dados = r.json().get('value', [])
        if dados:
            return dados[0]['cotacaoVenda']
        return None          # fim de semana / feriado
    except Exception:
        return None

# datas únicas de venda
datas_unicas = vendas['data_venda'].dt.date.unique()
print(f"Buscando câmbio para {len(datas_unicas)} datas...")

cambio_map = {}
for i, d in enumerate(sorted(datas_unicas)):
    taxa = buscar_cambio(str(d))
    cambio_map[str(d)] = taxa
    if i % 50 == 0:
        print(f"  {i}/{len(datas_unicas)} datas processadas...")
    time.sleep(0.15)   

# preenchendo fins de semana/feriados com o último câmbio disponível 
datas_sorted = sorted(cambio_map.keys())
ultima_taxa = None
for d in datas_sorted:
    if cambio_map[d] is not None:
        ultima_taxa = cambio_map[d]
    else:
        cambio_map[d] = ultima_taxa  

print("Câmbio carregado com sucesso!")

# juntando câmbio nas vendas
vendas['data_str']   = vendas['data_venda'].dt.strftime('%Y-%m-%d')
vendas['taxa_cambio'] = vendas['data_str'].map(cambio_map)

# juntando o custo unitário (USD) de cada produto
custos_sorted = custos.sort_values('data_inicial')

def custo_na_data(id_prod, data_venda):
    hist = custos_sorted[
        (custos_sorted['id_produto'] == id_prod) &
        (custos_sorted['data_inicial'] <= data_venda)
    ]
    if hist.empty:
        return None
    return hist.iloc[-1]['preco_usd']

print("Calculando custo unitário por transação (pode demorar)...")
vendas['custo_usd_unit'] = vendas.apply(
    lambda r: custo_na_data(r['id_produto'], r['data_venda']), axis=1
)

# calculando custo total BRL e prejuízo
vendas['custo_brl_total'] = vendas['custo_usd_unit'] * vendas['taxa_cambio'] * vendas['qtd']
vendas['prejuizo']        = (vendas['custo_brl_total'] - vendas['total']).clip(lower=0)

# salvando no SQLite para a query SQL 
conn = sqlite3.connect(":memory:")
vendas.to_sql('vendas_cambio', conn, if_exists='replace', index=False)

# agregando por id_produto 
query = """
SELECT
    id_produto,
    ROUND(SUM(total), 2)                                      AS receita_total,
    ROUND(SUM(prejuizo), 2)                                   AS prejuizo_total,
    ROUND(SUM(prejuizo) * 100.0 / NULLIF(SUM(total), 0), 4)  AS pct_perda
FROM vendas_cambio
WHERE custo_usd_unit IS NOT NULL
  AND taxa_cambio   IS NOT NULL
GROUP BY id_produto
HAVING SUM(prejuizo) > 0
ORDER BY pct_perda DESC;
"""
resultado = pd.read_sql(query, conn)
print(resultado.head(10))
print(f"\nProduto com MAIOR % de perda: id_produto = {resultado.iloc[0]['id_produto']}")
print(f"  % de perda: {resultado.iloc[0]['pct_perda']:.4f}%")

Buscando câmbio para 726 datas...
  0/726 datas processadas...
  50/726 datas processadas...
  100/726 datas processadas...
  150/726 datas processadas...
  200/726 datas processadas...
  250/726 datas processadas...
  300/726 datas processadas...
  350/726 datas processadas...
  400/726 datas processadas...
  450/726 datas processadas...
  500/726 datas processadas...
  550/726 datas processadas...
  600/726 datas processadas...
  650/726 datas processadas...
  700/726 datas processadas...
Câmbio carregado com sucesso!
Calculando custo unitário por transação (pode demorar)...
   id_produto  receita_total  prejuizo_total  pct_perda
0          72    63057815.65     39959694.85    63.3699
1          83    44377440.00     18382383.33    41.4228
2         109      328320.15       101206.61    30.8256
3         102      335379.20        96282.77    28.7086
4         136     1049801.00       251195.08    23.9279
5          47     3184863.80       680000.91    21.3510
6          31     466049

Questão 4.2: O produto com MAIOR % de perda: id_produto = 72.0

Questão 4.3: A cotação USD→BRL foi obtida da API PTAX do Banco Central, usando a venda do dia da transação. Para fins de semana e feriados, utilizei o último câmbio disponível.
Prejuízo: calculei o custo em reais de cada venda (preço em dólar × câmbio × quantidade). Se esse custo for maior que o valor da venda, consideramos a diferença como prejuízo. OBS: Transações sem custo disponível foram excluídas.

Questão 5:

In [104]:
vendas = pd.read_csv('../dados_tratados/vendas_crm_limpo.CSV')
produtos = pd.read_csv('../dados_tratados/produtos_crm_limpo.CSV')

In [105]:
vendas_produtos = vendas.merge(produtos[['id_produto','categoria_real']], on='id_produto', how='left')


In [106]:
df_vendas   = pd.read_csv('../dados_tratados/vendas_crm_limpo.csv')
df_produtos = pd.read_csv('../dados_tratados/produtos_crm_limpo.csv')

In [107]:
df_vendas.to_sql('vendas_crm_limpo', conn, if_exists='replace', index=False)
df_produtos.to_sql('produtos_crm_limpo', conn, if_exists='replace', index=False)

150

In [108]:
tabelas = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tabelas disponíveis:")
print(tabelas)


Tabelas disponíveis:
                 name
0       vendas_cambio
1    vendas_crm_limpo
2  produtos_crm_limpo


In [109]:
query = """
WITH categorias_limpas AS (
    SELECT
        id_produto,
        CASE
            WHEN LOWER(REPLACE(categoria_real, ' ', '')) IN (
                'eletronicos','eletrunicos','eletroniscos','eletronicoz'
            ) THEN 'eletronicos'
            WHEN LOWER(REPLACE(categoria_real, ' ', '')) IN (
                'propulsao','propulcao','prop','propulssao','propucao','propulsam'
            ) THEN 'propulsao'
            ELSE 'ancoragem'
        END AS categoria
    FROM produtos_crm_limpo
),
metricas_clientes AS (
    SELECT
        v.id_cliente,
        ROUND(SUM(v.total), 2)                         AS faturamento_total,
        COUNT(DISTINCT v.id)                           AS frequencia,
        ROUND(SUM(v.total) / COUNT(DISTINCT v.id), 2) AS ticket_medio,
        COUNT(DISTINCT c.categoria)                    AS diversidade_categorias
    FROM vendas_crm_limpo v
    JOIN categorias_limpas c ON v.id_produto = c.id_produto
    GROUP BY v.id_cliente
),
top10 AS (
    SELECT
        id_cliente,
        ticket_medio,
        diversidade_categorias
    FROM metricas_clientes
    WHERE diversidade_categorias >= 3
    ORDER BY ticket_medio DESC, id_cliente ASC
    LIMIT 10
)
SELECT
    c.categoria,
    SUM(v.qtd) AS total_itens
FROM vendas_crm_limpo v
JOIN categorias_limpas c ON v.id_produto = c.id_produto
WHERE v.id_cliente IN (SELECT id_cliente FROM top10)
GROUP BY c.categoria
ORDER BY total_itens DESC
LIMIT 1;
"""

resultado_q5 = pd.read_sql(query, conn)
print(resultado_q5)

   categoria  total_itens
0  propulsao         6030


Questão 6.1:

In [110]:
query = """
WITH RECURSIVE calendario AS (
    SELECT DATE(MIN(data_venda)) AS dia
    FROM vendas_crm_limpo

    UNION ALL

    SELECT DATE(dia, '+1 day')
    FROM calendario
    WHERE dia < (SELECT DATE(MAX(data_venda)) FROM vendas_crm_limpo)
),

calendario_ptbr AS (
    SELECT
        dia,
        CASE CAST(strftime('%w', dia) AS INTEGER)
            WHEN 0 THEN 'Domingo'
            WHEN 1 THEN 'Segunda-feira'
            WHEN 2 THEN 'Terca-feira'
            WHEN 3 THEN 'Quarta-feira'
            WHEN 4 THEN 'Quinta-feira'
            WHEN 5 THEN 'Sexta-feira'
            WHEN 6 THEN 'Sabado'
        END AS dia_semana,
        CAST(strftime('%w', dia) AS INTEGER) AS num_dia
    FROM calendario
),

vendas_diarias AS (
    SELECT
        c.dia,
        c.dia_semana,
        c.num_dia,
        COALESCE(SUM(v.total), 0) AS valor_venda
    FROM calendario_ptbr c
    LEFT JOIN vendas_crm_limpo v ON DATE(v.data_venda) = c.dia
    GROUP BY c.dia, c.dia_semana, c.num_dia
)

SELECT
    dia_semana,
    COUNT(dia)                 AS total_dias,
    ROUND(AVG(valor_venda), 2) AS media_vendas
FROM vendas_diarias
GROUP BY dia_semana, num_dia
ORDER BY media_vendas ASC;
"""

resultado_q6 = pd.read_sql(query, conn)
print(resultado_q6)

      dia_semana  total_dias  media_vendas
0        Domingo         105    3229614.16
1  Segunda-feira         105    3484500.47
2    Terca-feira         105    3488871.99
3   Quarta-feira         104    3534007.21
4   Quinta-feira         104    3713299.94
5         Sabado         104    3774290.79
6    Sexta-feira         104    3776151.25


Questão 6.2:
0        Domingo         105    3229614.16

Questão 6.3:
Usar uma tabela de datas é importante porque a tabela de vendas só registra os dias em que houve alguma venda. Se você fizer a análise direto nela, acaba ignorando os dias em que a loja não vendeu nada, o que distorce os resultados. Isso faz com que a média fique artificialmente mais alta, já que considera apenas os dias com venda.

Questão 7.1:

In [111]:
PRODUTO = 'Motor De Popa Yamaha Evo Dash 155Hp'   

vendas = pd.read_csv('../dados_tratados/vendas_crm_limpo.csv',
                     parse_dates=['data_venda'])
produtos = pd.read_csv('../dados_tratados/produtos_crm_limpo.csv')

# juntando para obter nome do produto
vp = vendas.merge(produtos[['id_produto', 'produto']], on='id_produto', how='left')
vp_filtrado = vp[vp['produto'].str.strip().str.title() == PRODUTO].copy()

# série diária
periodo_completo = pd.date_range(
    start=vp_filtrado['data_venda'].min(),
    end='2024-01-31',
    freq='D'
)
serie_diaria = (
    vp_filtrado.groupby('data_venda')['qtd'].sum()
    .reindex(periodo_completo, fill_value=0)
    .rename_axis('data')
    .reset_index(name='qtd_real')
)

# treino até 31/12/2023 | Teste = Janeiro 2024
treino = serie_diaria[serie_diaria['data'] <= '2023-12-31'].copy()
teste  = serie_diaria[serie_diaria['data'] >= '2024-01-01'].copy()

# média Móvel dos últimos 7 dias
historico_completo = serie_diaria.set_index('data')['qtd_real']

previsoes = []
for data in teste['data']:
    # usando apenas os 7 dias ANTERIORES à data prevista 
    janela = historico_completo[
        (historico_completo.index < data) &
        (historico_completo.index >= data - pd.Timedelta(days=7))
    ]
    previsao = janela.mean() if len(janela) > 0 else 0
    previsoes.append(round(previsao, 4))

teste = teste.copy()
teste['previsao'] = previsoes

# MAE 
mae = np.mean(np.abs(teste['qtd_real'] - teste['previsao']))

primeira_semana = teste[teste['data'] <= '2024-01-07']
soma_semana1 = round(primeira_semana['previsao'].sum())

print("=" * 55)
print(f"Produto analisado : {PRODUTO}")
print(f"MAE do modelo     : {mae:.4f} unidades/dia")
print(f"Soma prevista (01/01 a 07/01/2024): {soma_semana1} unidades")
print("=" * 55)
print("\nPrevisão diária - Janeiro 2024:")
print(teste[['data','qtd_real','previsao']].to_string(index=False))

Produto analisado : Motor De Popa Yamaha Evo Dash 155Hp
MAE do modelo     : 1.6406 unidades/dia
Soma prevista (01/01 a 07/01/2024): 3 unidades

Previsão diária - Janeiro 2024:
      data  qtd_real  previsao
2024-01-01         0    0.0000
2024-01-02         0    0.0000
2024-01-03         0    0.0000
2024-01-04         0    0.0000
2024-01-05        10    0.0000
2024-01-06         0    1.4286
2024-01-07         0    1.4286
2024-01-08         0    1.4286
2024-01-09         0    1.4286
2024-01-10         0    1.4286
2024-01-11         0    1.4286
2024-01-12         0    1.4286
2024-01-13         0    0.0000
2024-01-14         0    0.0000
2024-01-15         0    0.0000
2024-01-16         0    0.0000
2024-01-17         0    0.0000
2024-01-18         0    0.0000
2024-01-19         0    0.0000
2024-01-20         0    0.0000
2024-01-21        11    0.0000
2024-01-22         6    1.5714
2024-01-23         0    2.4286
2024-01-24         0    2.4286
2024-01-25         0    2.4286
2024-01-26        

Questão 7.2: Soma prevista (01/01 a 07/01/2024): 3 unidades

Questão 7.3: O baseline foi construído de forma simples: para cada dia de janeiro de 2024, a previsão é a média das vendas dos 7 dias anteriores, sempre usando apenas dados já observados e sem incluir o próprio dia ou qualquer informação futura. Para evitar vazamento de dados, garantimos que o cálculo considere apenas valores anteriores à data prevista, e o treino usa dados até 31/12/2023, com as previsões de janeiro sendo feitas dia a dia, sempre olhando para trás. Como limitação, esse modelo é bem básico e não capta padrões mais complexos, como sazonalidade ou tendências ao longo do tempo, então pode errar em períodos com comportamento diferente

Questão 8:

In [112]:
# matriz Usuário × Produto
matriz = (
    vendas.groupby(['id_cliente', 'id_produto'])['qtd']
    .sum()
    .unstack(fill_value=0)
    .clip(upper=1)
)

print(f"Matriz: {matriz.shape[0]} clientes × {matriz.shape[1]} produtos")

# similaridade de Cosseno com numpy
M = matriz.values.T.astype(float) 

# norma de cada produto
normas = np.linalg.norm(M, axis=1, keepdims=True)
normas[normas == 0] = 1  # evita divisão por zero

# normalizando matriz
M_norm = M / normas

# similaridade = produto interno dos vetores normalizados
sim_matrix = np.dot(M_norm, M_norm.T)

sim_df = pd.DataFrame(
    sim_matrix,
    index=matriz.columns,
    columns=matriz.columns
)

# identificando o id do produto de referência
PRODUTO_REF = 'Gps Garmin Vortex Maré Drift'

produtos['produto_norm'] = produtos['produto'].str.strip().str.title()

match = produtos[produtos['produto_norm'] == PRODUTO_REF]
if match.empty:
    match = produtos[produtos['produto_norm'].str.contains('Garmin Vortex', na=False)]

id_ref = match.iloc[0]['id_produto']
print(f"\nProduto de referência: '{match.iloc[0]['produto']}' (id_produto={id_ref})")

# top 5 mais similares 
similares = (
    sim_df[id_ref]
    .drop(index=id_ref)
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)
similares.columns = ['id_produto', 'similaridade']
similares = similares.merge(produtos[['id_produto', 'produto']], on='id_produto', how='left')

print("\n=== Top 5 produtos mais similares ===")
print(similares[['id_produto', 'produto', 'similaridade']].to_string(index=False))
print(f"\nResposta Q8.2 — id_produto com MAIOR similaridade: {similares.iloc[0]['id_produto']}")
print(f"  Produto : {similares.iloc[0]['produto']}")
print(f"  Score   : {similares.iloc[0]['similaridade']:.6f}")

Matriz: 49 clientes × 150 produtos

Produto de referência: 'Gps Garmin Vortex Maré Drift' (id_produto=27)

=== Top 5 produtos mais similares ===
 id_produto                                    produto  similaridade
         94           Motor De Popa Volvo Magnum 276Hp      0.869626
         11        Gps Furuno Swift Leviathan Poseidon      0.868037
         35                         Radar Furuno Swift      0.853913
          1                Transponder Ais Maré Magnum      0.850000
        115 Cabo De Nylon Delta Force Magnum Leviathan      0.850000

Resposta Q8.2 — id_produto com MAIOR similaridade: 94
  Produto : Motor De Popa Volvo Magnum 276Hp
  Score   : 0.869626
